In [1]:
# generate duties

# importing the packages
import pandas as pd
from datetime import datetime, timedelta


def generate_duties_dfs(df, current_duty, valid_duty):
    # Recursive DFS for each remaining flight leg
    if is_valid_duty(df, current_duty):
        valid_duty.append(current_duty.copy())
        return
    if len(current_duty) == 3:
        return

    # Recursive DFS for each remaining flight leg
    for flight_leg_id in df[df["departure_airport"] == df.loc[df["flight_leg_id"] == current_duty[-1], "destination_airport"].values[0]]['flight_leg_id']: # Space Constraint
        # Check if the leg can be added to the current pair
        # if is_valid_duty(df, current_duty.copy() + [flight_leg_id]):
        if sub_df(df, current_duty, flight_leg_id):
            current_duty.append(flight_leg_id)
            generate_duties_dfs(df, current_duty, valid_duty)
            current_duty.pop()  # Backtrack

# function taking care of the time constraint and total time constraint
def sub_df(df, current_duty, flight_leg_id):
    return (((
                datetime.strptime(
                    df.loc[
                        df["flight_leg_id"] == flight_leg_id, "start_time"
                    ].values[0],
                    "%Y-%m-%d %H:%M:%S",
                )
                - datetime.strptime(
                    df.loc[df["flight_leg_id"] == current_duty[-1], "end_time"].values[
                        0
                    ],
                    "%Y-%m-%d %H:%M:%S",
                )
            )
            >= timedelta(hours=1)) and (
                datetime.strptime(
                    df.loc[
                        df["flight_leg_id"] == flight_leg_id, "start_time"
                    ].values[0],
                    "%Y-%m-%d %H:%M:%S",
                )
                - datetime.strptime(
                    df.loc[df["flight_leg_id"] == current_duty[0], "start_time"].values[
                        0
                    ],
                    "%Y-%m-%d %H:%M:%S",
                )
            )
            <= timedelta(hours=10))

# total time constraint and return to homebase constraint
def is_valid_duty(df, current_duty):
    # Implement conditions for pair validity
    # Ensure matching destination and departure airports, and time gaps
    return (
        (
            df.loc[
                df["flight_leg_id"] == current_duty[-1], "destination_airport"
            ].values[0]
            == df.loc[
                df["flight_leg_id"] == current_duty[0], "departure_airport"
            ].values[0]
        ) and (
            datetime.strptime(
                df.loc[
                    df["flight_leg_id"] == current_duty[-1], "start_time"
                ].values[0],
                "%Y-%m-%d %H:%M:%S",
            )
            - datetime.strptime(
                df.loc[df["flight_leg_id"] == current_duty[0], "start_time"].values[
                    0
                ],
                "%Y-%m-%d %H:%M:%S",
            )
        )
        <= timedelta(hours=10))


def generate_duties():
    # reading the dataframe
    df = pd.read_csv("../data/flight_legs/data.csv")

    # Create an empty list to hold the duties
    valid_duty_dfs = []

    # Start DFS from each flight leg
    for start_leg in df["flight_leg_id"]:
        generate_duties_dfs(df, [start_leg], valid_duty_dfs)

    # Open the file in write mode
    with open(f"../data/duties/duties.txt", "w") as file:
        # Write each item in the list to a new line
        for item in valid_duty_dfs:
            file.write(f"{item}\n")
            
generate_duties()

In [3]:
# numpy code 
# generate duties

# importing the packages
import numpy as np
from numba import jit, cuda
import pandas as pd
from datetime import timedelta
from itertools import chain
from warnings import filterwarnings

filterwarnings("ignore")


def generate_duties_dfs(np_arr, current_duty, valid_duty):
    # Recursive DFS for each remaining flight leg
    if is_valid_duty(np_arr, current_duty):
        valid_duty.append(current_duty.copy())
        return
    if len(current_duty) == 3:
        return

    # Recursive DFS for each remaining flight leg
    #[slist for slist in range(len(np_arr)) if (np_arr[slist][1] == np_arr[current_duty[-1]][2])]
    #np.arange(len(np_arr))[np_arr[:,1] == np_arr[current_duty[-1]][2]]
    for index in np.where(np_arr[:,1] == np_arr[current_duty[-1]][2])[0]:
        # Check if the leg can be added to the current pair
        if sub_df(np_arr, current_duty, index):
            current_duty.append(index)
            generate_duties_dfs(np_arr, current_duty, valid_duty)
            current_duty.pop()  # Backtrack

# function taking care of the time constraint and total time constraint
@jit(nopython=False)
def sub_df(np_arr, current_duty, index):
    return (
        ((np_arr[index][3] - np_arr[current_duty[-1]][4] >= timedelta(hours=1)))
        and
        ((np_arr[index][3] - np_arr[current_duty[-1]][4] <= timedelta(hours=3)))
    )

# total time constraint and return to homebase constraint
def is_valid_duty(np_arr, current_duty):
    # Implement conditions for pair validity
    # Ensure matching destination and departure airports, and time gaps
    return (
        (np_arr[current_duty[0]][1] == np_arr[current_duty[-1]][2])  # return to homebase condition
        and 
        ((np_arr[current_duty[-1]][4] - np_arr[current_duty[0]][3]) <= timedelta(hours=12))
    )

def generate_duties():
    # reading the dataframe
    df = pd.read_csv("../data/flight_legs/data.csv")
    df['start_time'] = pd.to_datetime(df['start_time'], format="%Y-%m-%d %H:%M:%S")
    df['end_time'] = pd.to_datetime(df['end_time'], format="%Y-%m-%d %H:%M:%S")

    # convert the dataframe to a numpy array
    np_arr = df.to_numpy()

    # Create an empty list to hold the duties
    valid_duty_dfs = []

    # Start DFS from each flight leg
    for start_leg in range(len(np_arr)):
        generate_duties_dfs(np_arr, [start_leg], valid_duty_dfs)

    # Open the file in write mode
    with open(f"../data/duties/duties.txt", "w") as file:
        # Write each item in the list to a new line
        for item in valid_duty_dfs:
            file.write(f"{item}\n")

generate_duties()

KeyboardInterrupt: 

In [37]:
# importing the packages
import numpy as np
import pandas as pd
from warnings import filterwarnings

filterwarnings("ignore")

def generate_duties_dfs(np_arr, current_duty, valid_duty):
    if is_valid_duty(np_arr, current_duty):
        valid_duty.append(current_duty.copy().tolist())
        return
    if len(current_duty) == 2:
        return

    # Recursive DFS for each remaining flight leg
    for index in np.where(np_arr[:,1] == np_arr[current_duty[-1]][2])[0]:
        # Check if the leg can be added to the current pair
        if sub_df(np_arr, current_duty, index):
            current_duty = np.append(current_duty, index)
            generate_duties_dfs(np_arr, current_duty, valid_duty)
            current_duty = current_duty[:-1]  # Backtrack

# function taking care of the time constraint and total time constraint
def sub_df(np_arr, current_duty, index):
    time_difference = np_arr[index][3] - np_arr[current_duty[-1]][4]
    return (
        ((time_difference >= np.timedelta64(1, 'h')))
        and
        ((time_difference <= np.timedelta64(3, 'h')))
    )

# total time constraint and return to homebase constraint
def is_valid_duty(np_arr, current_duty):
    # Implement conditions for pair validity
    return (
        (np_arr[current_duty[0]][1] == np_arr[current_duty[-1]][2])  # return to homebase condition
        and 
        ((np_arr[current_duty[-1]][4] - np_arr[current_duty[0]][3]) <= np.timedelta64(8, 'h'))
    )

def generate_duties():
    # reading the dataframe
    df = pd.read_csv("../data/flight_legs/data.csv")
    df['start_time'] = pd.to_datetime(df['start_time'], format="%Y-%m-%d %H:%M:%S")
    df['end_time'] = pd.to_datetime(df['end_time'], format="%Y-%m-%d %H:%M:%S")

    # convert the dataframe to a numpy array
    np_arr = df.to_numpy()

    # Create an empty list to hold the duties
    valid_duty_dfs = []

    # Start DFS from each flight leg
    for start_leg in range(len(np_arr)):
        generate_duties_dfs(np_arr, np.array([start_leg]), valid_duty_dfs)

    # Open the file in write mode
    with open(f"../data/duties/duties.txt", "w") as file:
        file.writelines(f"{item}\n" for item in valid_duty_dfs)
    

generate_duties()

In [38]:
# Open the file in read mode
with open("../data/duties/duties.txt", "r") as file:
    lines = file.readlines()

# convert each line (representing a list) to an actual list
duties = [eval(line) for line in lines]

from itertools import chain
duty = list(set(chain(*duties)))

len(duties), len(duty)

(1701, 240)